In [0]:
"""Install required packages.

Installs:
- mlflow: Latest version for MLflow tracing (auto-captures LLM calls)
- pyrepo-mcda: For CRITIC weighting method

Restarts Python kernel to activate installed packages.
"""

%pip install -U mlflow
%pip install pyrepo-mcda

dbutils.library.restartPython()

In [0]:
"""Import libraries and define notebook constants."""

from pyspark.sql import functions as F, types as T, DataFrame
from pyspark.sql.window import Window
from datetime import datetime
import pandas as pd
import numpy as np
import json
from openai import OpenAI
from pyrepo_mcda import weighting_methods as mcda_weights
import mlflow

mlflow.openai.autolog()

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
CATALOG = "primeins"
SCHEMA_S = f"{CATALOG}.silver"
SCHEMA_G = f"{CATALOG}.gold"

In [0]:
"""PrimeInsurance Claims Risk & Anomaly Engine.

Catalog: primeins
Source: primeins.silver.silver_claims
Output: primeins.gold.claim_anomaly_explanations

Approach:
---------
Detection: Pure statistics (z-scores, IQR, count-based rules).
           LLM is only used to narrate findings, not for detection.

Weighting: CRITIC method (objective weighting based on variance and correlation)
           Assigns higher weights to rules with:
           (a) HIGH contrast (large std-dev)
           (b) LOW correlation with other rules (unique signal)

Anomaly Rules:
--------------
R1: Amount Z-Score      - Total amount far above population mean
R2: Severity Mismatch   - Amount unusually high for severity level
R3: Injury/Vehicle Ratio - Injury payout disproportionate to vehicle damage
R4: Staged Accident     - Multi-vehicle + no witnesses + no police
R5: Documentation Gap   - Missing property_damage, police_report, or authorities

Thresholds:
-----------
HIGH:   score >= 65
MEDIUM: score >= 40 and < 65
LOW:    score < 40

Output: All claims written to primeins.gold.claim_anomaly_explanations
        Flagged claims (HIGH/MEDIUM) receive AI investigation briefs.
"""

# Define thresholds
ANOMALY_THRESHOLD = 40
HIGH_THRESHOLD = 65

In [0]:
"""Configure LLM client for AI investigation brief generation.

TEST_MODE = True:  OpenRouter free tier (no Databricks billing)
TEST_MODE = False: Databricks Foundation Model endpoint (production)
"""

TEST_MODE = False

# api_key = dbutils.secrets.get(scope="my_secrets", key="openrouter_api_key")
gemini_api_key = dbutils.secrets.get(scope="gemini_secrets", key="gemini_api_key")

if TEST_MODE:
    # TEST: OpenRouter free tier
    client = OpenAI(
        api_key=api_key,
        base_url="https://openrouter.ai/api/v1",
    )
    MODEL_ID = "liquid/lfm-2.5-1.2b-instruct:free"
    TARGET_TABLE = f"{SCHEMA_G}.claim_anomaly_explanations"
    print("⚠️  TEST MODE — using OpenRouter free tier")
else:
    # PRODUCTION: Databricks Foundation Model
    DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
    WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl")
    client = OpenAI(
        api_key=DATABRICKS_TOKEN,
        base_url=f"https://{WORKSPACE_URL}/serving-endpoints",
    )
    MODEL_ID = "databricks-gpt-oss-20b"
    TARGET_TABLE = f"{SCHEMA_G}.claim_anomaly_explanations"
    print("🚀  PRODUCTION MODE — using Databricks Foundation Model")

print(f"Model    : {MODEL_ID}")
print(f"Output   : {TARGET_TABLE}")
print(f"Run ID   : {RUN_ID}")
print(f"Schema   : {SCHEMA_G}")

In [0]:
"""Helper functions for table operations."""


def write_gold(df: DataFrame, table: str, comment: str = "") -> int:
    """Write DataFrame to Gold table with full overwrite.
    
    Args:
        df: Spark DataFrame to write
        table: Table name (without schema prefix)
        comment: Optional comment to print
    
    Returns:
        Row count of written table
    """
    full = f"{SCHEMA_G}.{table}"
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(full))
    cnt = spark.table(full).count()
    print(f"✅  {full:<60} {cnt:>8,} rows  {comment}")
    return cnt


def read_current(table: str) -> DataFrame:
    """Read current rows from SCD-2 table.
    
    Filters on __END_AT IS NULL to get only active records.
    
    Args:
        table: Fully qualified table name
    
    Returns:
        DataFrame with only current rows
    """
    df = spark.table(table)
    if "__END_AT" in df.columns:
        df = df.filter(F.col("__END_AT").isNull()).drop("__START_AT", "__END_AT")
    return df

In [0]:
"""Load silver claims and enrich with policy/customer dimensions.

Steps:
1. Load silver_claims (current rows only)
2. LEFT JOIN to dim_policy (adds customer_id, policy_csl)
3. LEFT JOIN to dim_customer (adds customer_region)
4. Cast numeric columns and derive computed fields
"""

# Load silver claims
claims_raw = read_current(f"{SCHEMA_S}.silver_claims")
print(f"silver_claims loaded: {claims_raw.count():,} rows")
print("Columns:", claims_raw.columns)

# Enrich with policy data (customer_id, policy_csl)
try:
    dim_pol = read_current(f"{SCHEMA_G}.dim_policy")
    pol_cols_needed = [c for c in ["policy_number", "customer_id", "policy_csl"]
                       if c in dim_pol.columns]
    print(f"dim_policy available -- columns used: {pol_cols_needed}")

    if "policy_number" in dim_pol.columns:
        dim_pol_slim = (dim_pol
            .select(*pol_cols_needed)
            .withColumnRenamed("policy_number", "policyid")
        )
        claims_raw = claims_raw.join(dim_pol_slim, on="policyid", how="left")
        print("✅ Joined dim_policy -> customer_id and policy_csl added")
    else:
        raise ValueError("policy_number column missing from dim_policy")
except Exception as e:
    print(f"⚠️  WARNING dim_policy not available -- skipping: {e}")
    for stub in ["customer_id", "policy_csl"]:
        if stub not in claims_raw.columns:
            claims_raw = claims_raw.withColumn(stub, F.lit(None).cast(T.StringType()))

# Enrich with customer data (region)
try:
    dim_cust = read_current(f"{SCHEMA_G}.dim_customer")
    print(f"dim_customer available -- columns: {dim_cust.columns}")

    if "customer_id" in dim_cust.columns and "region" in dim_cust.columns:
        dim_cust_slim = (dim_cust
            .select("customer_id", F.col("region").alias("customer_region"))
        )
        claims_raw = claims_raw.join(dim_cust_slim, on="customer_id", how="left")
        print("✅ Joined dim_customer -> customer_region added")
    else:
        raise ValueError("customer_id or region column missing from dim_customer")
except Exception as e:
    print(f"⚠️  WARNING dim_customer not available -- skipping: {e}")
    if "customer_region" not in claims_raw.columns:
        claims_raw = claims_raw.withColumn("customer_region", F.lit(None).cast(T.StringType()))

print(f"\nPost-enrichment columns: {claims_raw.columns}")

# Cast numeric columns (use try_cast to handle malformed data gracefully)
amount_cols = ["injury", "vehicle", "property"]
for nc in amount_cols:
    if nc in claims_raw.columns:
        claims_raw = claims_raw.withColumn(
            nc, F.expr(f"try_cast(`{nc}` as double)")
        )

other_numeric = ["witnesses", "number_of_vehicles_involved", "bodily_injuries"]
for nc in other_numeric:
    if nc in claims_raw.columns:
        claims_raw = claims_raw.withColumn(
            nc, F.expr(f"try_cast(`{nc}` as double)")
        )

# Derive total_claim_amount
claims_raw = claims_raw.withColumn(
    "total_claim_amount",
    F.coalesce(F.col("injury"), F.lit(0.0)) +
    F.coalesce(F.col("vehicle"), F.lit(0.0)) +
    F.coalesce(F.col("property"), F.lit(0.0))
)
print("Derived total_claim_amount = injury + vehicle + property")

# Derive claim_processing_days
if "claim_processing_days" not in claims_raw.columns:
    if "claim_logged_on" in claims_raw.columns and "claim_processed_on" in claims_raw.columns:
        for dc in ["claim_logged_on", "claim_processed_on", "incident_date"]:
            if dc in claims_raw.columns:
                claims_raw = claims_raw.withColumn(
                    dc, F.expr(f"try_to_date(`{dc}`, 'yyyy-MM-dd')")
                )
        claims_raw = claims_raw.withColumn(
            "claim_processing_days",
            F.datediff(F.col("claim_processed_on"), F.col("claim_logged_on")).cast(T.DoubleType())
        )
        print("Derived claim_processing_days = claim_processed_on - claim_logged_on")
    else:
        claims_raw = claims_raw.withColumn(
            "claim_processing_days", F.lit(None).cast(T.DoubleType())
        )
        print("claim_processing_days: could not derive -- date columns missing")
else:
    for dc in ["claim_logged_on", "claim_processed_on", "incident_date"]:
        if dc in claims_raw.columns:
            claims_raw = claims_raw.withColumn(
                dc, F.expr(f"try_to_date(`{dc}`, 'yyyy-MM-dd')")
            )
    claims_raw = claims_raw.withColumn(
        "claim_processing_days",
        F.expr("try_cast(claim_processing_days as double)")
    )

# Alias component columns to standard names
claims_raw = (claims_raw
    .withColumn("injury_claim", F.col("injury"))
    .withColumn("vehicle_claim", F.col("vehicle"))
    .withColumn("property_claim", F.col("property"))
)

claims = claims_raw
total_claims = claims.count()
print(f"\nClaims ready for scoring: {total_claims:,}")
print(f"Sample totals -- max: {claims.agg(F.max('total_claim_amount')).collect()[0][0]:,.0f}  "
      f"mean: {claims.agg(F.mean('total_claim_amount')).collect()[0][0]:,.0f}")

In [0]:
"""Population statistics for anomaly scoring.

Computes population-level metrics used as reference points:
- Amount statistics (mean, std, IQR) for all claims
- Severity-level statistics (mean, std) per incident severity
- Claim counts per customer/policy
"""

# Amount statistics for R1 (Amount Z-Score)
amt_stats = (claims
    .agg(
        F.mean("total_claim_amount").alias("amt_mean"),
        F.stddev_pop("total_claim_amount").alias("amt_std"),
        F.expr("percentile_approx(total_claim_amount, 0.25)").alias("amt_q1"),
        F.expr("percentile_approx(total_claim_amount, 0.75)").alias("amt_q3"),
    )
    .collect()[0]
)
AMT_MEAN = float(amt_stats["amt_mean"] or 0)
AMT_STD  = float(amt_stats["amt_std"]  or 1)
AMT_IQR  = float((amt_stats["amt_q3"] or 0) - (amt_stats["amt_q1"] or 0))
print(f"Amount  -> mean={AMT_MEAN:,.0f}  std={AMT_STD:,.0f}  IQR={AMT_IQR:,.0f}")

# Severity-level statistics for R2 (Severity Mismatch)
sev_stats = (claims
    .filter(F.col("incident_severity").isNotNull())
    .groupBy("incident_severity")
    .agg(
        F.mean("total_claim_amount").alias("sev_mean"),
        F.stddev_pop("total_claim_amount").alias("sev_std"),
    )
)
sev_stats.show(truncate=False)

sev_dict = {
    row["incident_severity"]: (
        float(row["sev_mean"] or 0),
        float(row["sev_std"]  or 1),
    )
    for row in sev_stats.collect()
}
print("Severity map:", sev_dict)

# Customer claim counts (used in LLM prompt context)
if "customer_id" in claims.columns and claims.filter(F.col("customer_id").isNotNull()).count() > 0:
    repeat_key = "customer_id"
else:
    repeat_key = "policyid"
    print(f"WARNING: customer_id unavailable -- using {repeat_key} for claim count")

if "customer_claim_count" in claims.columns:
    claims = claims.drop("customer_claim_count")

repeat_counts = (claims
    .groupBy(repeat_key)
    .agg(F.count("claimid").alias("customer_claim_count"))
)

claims = claims.join(
    F.broadcast(repeat_counts),
    on=repeat_key,
    how="left"
)

claims = claims.withColumn(
    "customer_claim_count",
    F.coalesce(F.col("customer_claim_count"), F.lit(1))
)

high_freq = claims.filter(F.col("customer_claim_count") > F.lit(3)).select(repeat_key).distinct().count()
print(f"Claim count keyed on : {repeat_key}")
print(f"High-frequency (>3)  : {high_freq}")

In [0]:
display(claims)

In [0]:
"""R1 - Amount Z-Score.

Detects claims with inflated amounts relative to population.
Computes z-score of total_claim_amount, capped at z=2, normalized to [0,1].
"""

claims = claims.withColumn(
    "_r1_zscore_raw",
    F.when(
        F.col("total_claim_amount").isNotNull(),
        (F.col("total_claim_amount") - F.lit(AMT_MEAN)) / F.lit(AMT_STD)
    ).otherwise(F.lit(0.0))
)

claims = claims.withColumn(
    "r1_amount_zscore",
    F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("_r1_zscore_raw") / F.lit(2.0)))
)

In [0]:
"""R2 - Severity Mismatch.

Detects claims with amounts unusually high within their severity category.
A "Trivial" claim priced like "Major" suggests intentional mislabeling.
Uses within-severity z-score, capped at z=2, normalized to [0,1].
"""

sev_mean_expr = F.lit(AMT_MEAN)
sev_std_expr  = F.lit(AMT_STD)

for sev, (m, s) in sev_dict.items():
    sev_mean_expr = F.when(F.col("incident_severity") == sev, F.lit(m)).otherwise(sev_mean_expr)
    sev_std_expr  = F.when(F.col("incident_severity") == sev, F.lit(max(s, 1.0))).otherwise(sev_std_expr)

claims = claims.withColumn(
    "_r2_sev_z",
    F.when(
        F.col("total_claim_amount").isNotNull(),
        (F.col("total_claim_amount") - sev_mean_expr) / sev_std_expr
    ).otherwise(F.lit(0.0))
)

claims = claims.withColumn(
    "r2_severity_mismatch",
    F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("_r2_sev_z") / F.lit(2.0)))
)

In [0]:
"""R3 - Injury/Vehicle Ratio Anomaly.

Detects disproportionate injury payouts vs vehicle damage.
Fraudsters inflate injury claims as they're harder to verify.
Computes z-score of injury/(vehicle+1) ratio, normalized to [0,1].
"""

ratio_stats = (claims
    .withColumn("_inj_veh_ratio",
        F.col("injury_claim") / (F.col("vehicle_claim") + F.lit(1.0))
    )
    .agg(
        F.mean("_inj_veh_ratio").alias("ratio_mean"),
        F.stddev_pop("_inj_veh_ratio").alias("ratio_std"),
    )
    .collect()[0]
)

RATIO_MEAN = float(ratio_stats["ratio_mean"] or 0)
RATIO_STD  = float(ratio_stats["ratio_std"]  or 1)
print(f"Injury/Vehicle ratio -> mean={RATIO_MEAN:.3f}  std={RATIO_STD:.3f}")

claims = claims.withColumn(
    "_r3_ratio_z",
    F.when(
        F.col("injury_claim").isNotNull() & F.col("vehicle_claim").isNotNull(),
        (
            (F.col("injury_claim") / (F.col("vehicle_claim") + F.lit(1.0)))
            - F.lit(RATIO_MEAN)
        ) / F.lit(max(RATIO_STD, 0.001))
    ).otherwise(F.lit(0.0))
)

claims = claims.withColumn(
    "r3_injury_ratio",
    F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("_r3_ratio_z") / F.lit(2.0)))
)

In [0]:
"""R4 & R5 - Binary Flag Rules.

R4 - Staged Accident: Multi-vehicle collision with no witnesses and no police.
     Real multi-car accidents almost always involve witnesses/law enforcement.
     Score: (vehicles>1 + witnesses==0 + no_police) / 3 -> [0, 0.33, 0.67, 1.0]

R5 - Documentation Gap: Missing documentation is a hallmark of staged claims.
     Score: (property_damage NULL + no police report + authorities==None) / 3
"""

# R4 - Staged Accident
has_nv = "number_of_vehicles_involved" in claims.columns
has_wi = "witnesses" in claims.columns
has_pr = "police_report_available" in claims.columns

c1 = (F.col("number_of_vehicles_involved") > F.lit(1)).cast(T.DoubleType()) if has_nv else F.lit(0.0)
c2 = (F.col("witnesses") == F.lit(0)).cast(T.DoubleType()) if has_wi else F.lit(0.0)
c3 = (F.upper(F.col("police_report_available")) != F.lit("YES")).cast(T.DoubleType()) if has_pr else F.lit(0.0)

claims = claims.withColumn(
    "r4_staged_accident",
    F.round(
        (F.coalesce(c1, F.lit(0.0)) +
         F.coalesce(c2, F.lit(0.0)) +
         F.coalesce(c3, F.lit(0.0))) / F.lit(3.0),
        4
    )
)

# R5 - Documentation Gap
has_pd = "property_damage" in claims.columns
has_auth = "authorities_contacted" in claims.columns

s1 = F.col("property_damage").isNull().cast(T.DoubleType()) if has_pd else F.lit(0.0)
s2 = (
    F.col("police_report_available").isNull() |
    (F.upper(F.col("police_report_available")) != F.lit("YES"))
).cast(T.DoubleType()) if has_pr else F.lit(0.0)
s3 = (F.lower(F.col("authorities_contacted")) == F.lit("none")).cast(T.DoubleType()) if has_auth else F.lit(0.0)

claims = claims.withColumn(
    "r5_doc_gap",
    F.round(
        (F.coalesce(s1, F.lit(0.0)) +
         F.coalesce(s2, F.lit(0.0)) +
         F.coalesce(s3, F.lit(0.0))) / F.lit(2.0),
        4
    )
)

In [0]:
display(claims)

In [0]:
"""Derive objective weights using CRITIC method.

CRITIC (CRiteria Importance Through Intercriteria Correlation)
assigns higher weights to rules with:
- HIGH variance (discriminatory power)
- LOW correlation (unique, non-redundant signal)

Weights are data-driven, not manually assigned, making them
defensible under regulatory audit.
"""

INDICATOR_COLS = [
    "r1_amount_zscore",
    "r2_severity_mismatch",
    "r3_injury_ratio",
    "r4_staged_accident",
    "r5_doc_gap",
]

# Collect indicator matrix (5 cols x ~3000 rows)
indicator_pdf = (claims
    .select(*INDICATOR_COLS)
    .fillna(0)
    .toPandas()
)

matrix = indicator_pdf.to_numpy()

# ── CRITIC internals (for transparency) ─────────────────────────────
stds        = matrix.std(axis=0)
corr_matrix = np.corrcoef(matrix.T)
conflict    = np.array([
    sum(1 - abs(corr_matrix[i][j]) for j in range(len(INDICATOR_COLS)) if j != i)
    for i in range(len(INDICATOR_COLS))
])
info_measures = stds * conflict

# ── Derive weights ───────────────────────────────────────────────────
# NOTE: pymcdm function is critic_weights (not critic_weighting)
W = mcda_weights.critic_weighting(matrix)

# ── Print: Step 1 — Standard Deviations ─────────────────────────────
print("\n" + "=" * 60)
print("CRITIC METHOD — STEP 1: DISCRIMINATORY POWER (STD DEV)")
print("Higher std dev = more variance = stronger signal across claims")
print("=" * 60)
print(f"{'Rule':<28} {'Std Dev':>10}  {'Interpretation'}")
print("-" * 60)
for col, s in zip(INDICATOR_COLS, stds):
    flag = "← highest variance" if s == stds.max() else ""
    print(f"{col:<28} {s:>10.4f}  {flag}")

# ── Print: Step 2 — Correlation Matrix ──────────────────────────────
print("\n" + "=" * 60)
print("CRITIC METHOD — STEP 2: INTERCRITERIA CORRELATION MATRIX")
print("High correlation = redundant signal = weight penalty applied")
print("=" * 60)
labels = ["R1", "R2", "R3", "R4", "R5"]
header = f"{'':>6}" + "".join(f"{l:>8}" for l in labels)
print(header)
print("-" * (6 + 8 * len(labels)))
for i, row_label in enumerate(labels):
    row = f"{row_label:>6}"
    for j in range(len(labels)):
        val = corr_matrix[i][j]
        marker = " ⚠" if i != j and abs(val) > 0.9 else "  "
        row += f"{val:>7.3f}{marker[1]}"
    print(row)
print("\n⚠ = correlation > 0.90 (near-redundant signals, weight reduced by CRITIC)")

# ── Print: Step 3 — Conflict Measure ────────────────────────────────
print("\n" + "=" * 60)
print("CRITIC METHOD — STEP 3: CONFLICT MEASURE")
print("Conflict = sum of (1 - |r_ij|) across all other rules")
print("Low conflict (correlated with others) → downweighted")
print("=" * 60)
print(f"{'Rule':<28} {'Conflict':>10}  {'Note'}")
print("-" * 60)
for col, c in zip(INDICATOR_COLS, conflict):
    note = "← penalised (collinear with R1)" if col == "r2_severity_mismatch" else \
           "← penalised (collinear with R2)" if col == "r1_amount_zscore" else \
           "← independent signal, rewarded"  if c == conflict.max() else ""
    print(f"{col:<28} {c:>10.4f}  {note}")

# ── Print: Step 4 — Information Measures ────────────────────────────
print("\n" + "=" * 60)
print("CRITIC METHOD — STEP 4: INFORMATION MEASURE")
print("Info = Std Dev × Conflict  (variance × uniqueness)")
print("This is the raw score before normalisation to weights")
print("=" * 60)
print(f"{'Rule':<28} {'Std Dev':>9} {'Conflict':>10} {'Info':>8}")
print("-" * 60)
for col, s, c, inf in zip(INDICATOR_COLS, stds, conflict, info_measures):
    print(f"{col:<28} {s:>9.4f} {c:>10.4f} {inf:>8.4f}")

# ── Print: Final Weights ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("CRITIC METHOD — FINAL WEIGHTS (normalised)")
print("=" * 60)
print(f"{'Rule':<28} {'Weight':>8}  {'Score contribution (×100)'}")
print("-" * 60)
for col, w in zip(INDICATOR_COLS, W):
    bar = "█" * int(w * 40)
    print(f"{col:<28} {w:>8.4f}  {bar}")
print(f"\n{'Weights sum to:':<28} {W.sum():>8.6f}")

print("\nKEY INSIGHT: R1 and R2 are 99.9% correlated — CRITIC")
print("automatically penalises both via the conflict measure.")
print("R4 and R5 earn higher weights due to low inter-rule")
print("correlation (unique signal), despite lower std devs.")

w1, w2, w3, w4, w5 = [float(x) for x in W]

In [0]:
"""Compute weighted composite anomaly score.

Combines 5 rule indicators using CRITIC-derived weights.
Score = 100 * sum(w_j * indicator_j) in range [0, 100]

Tiers:
- HIGH: score >= 65
- MEDIUM: score >= 40 and < 65
- LOW: score < 40
"""

claims = claims.withColumn(
    "anomaly_score",
    F.round(
        F.lit(100.0) * (
            F.lit(w1) * F.col("r1_amount_zscore") +
            F.lit(w2) * F.col("r2_severity_mismatch") +
            F.lit(w3) * F.col("r3_injury_ratio") +
            F.lit(w4) * F.col("r4_staged_accident") +
            F.lit(w5) * F.col("r5_doc_gap")
        ),
        2
    )
)

# Assign priority tier
claims = claims.withColumn(
    "priority_tier",
    F.when(F.col("anomaly_score") >= F.lit(HIGH_THRESHOLD), F.lit("HIGH"))
     .when(F.col("anomaly_score") >= F.lit(ANOMALY_THRESHOLD), F.lit("MEDIUM"))
     .otherwise(F.lit("LOW"))
)

# Human-readable triggered rules string
claims = claims.withColumn(
    "triggered_rules",
    F.concat_ws(", ",
        F.when(F.col("r1_amount_zscore") >= 0.4, F.lit("R1:HighAmount")).otherwise(F.lit(None)),
        F.when(F.col("r2_severity_mismatch") >= 0.4, F.lit("R2:SeverityMismatch")).otherwise(F.lit(None)),
        F.when(F.col("r3_injury_ratio") >= 0.4, F.lit("R3:InjuryRatio")).otherwise(F.lit(None)),
        F.when(F.col("r4_staged_accident") >= 0.4, F.lit("R4:StagedAccident")).otherwise(F.lit(None)),
        F.when(F.col("r5_doc_gap") >= 0.3, F.lit("R5:DocGap")).otherwise(F.lit(None)),
    )
)

# Summary statistics
claims.groupBy("priority_tier").count().orderBy("priority_tier").show()
flagged_count = claims.filter(F.col("priority_tier").isin(["HIGH", "MEDIUM"])).count()
print(f"Total: {total_claims:,}  |  Flagged: {flagged_count:,}  ({100*flagged_count/total_claims:.1f}%)")

In [0]:
"""Business rule driven weight assignment.

Domain experts assign weights based on fraud investigation experience:
- R1 (Amount Z-Score)       : 30% — inflated amounts are the #1 direct signal
- R2 (Severity Mismatch)    : 25% — severity downgrading is a known evasion tactic
- R3 (Injury/Vehicle Ratio) : 20% — injury inflation is hard to verify, common in fraud
- R4 (Staged Accident)      : 15% — strong signal but only fires on multi-vehicle claims
- R5 (Documentation Gap)    : 10% — weakest alone, but amplifies other signals

Note: R1 and R2 intentionally receive high weights because amount-based signals
are the primary trigger in 70%+ of auto insurance fraud cases (industry benchmark).
Unlike CRITIC, business weights do NOT penalise R1/R2 collinearity — investigators
want financial anomalies double-weighted because both angles matter in court.
"""

BW = {
    "r1_amount_zscore":    0.30,
    "r2_severity_mismatch": 0.25,
    "r3_injury_ratio":      0.20,
    "r4_staged_accident":   0.15,
    "r5_doc_gap":           0.10,
}

# Verify weights sum to 1
assert abs(sum(BW.values()) - 1.0) < 1e-9, "Business weights must sum to 1"

bw1, bw2, bw3, bw4, bw5 = [BW[c] for c in INDICATOR_COLS]

print("\n=== BUSINESS RULE WEIGHTS vs CRITIC WEIGHTS ===")
print(f"{'Rule':<28} {'Business Wt':>12} {'CRITIC Wt':>12} {'Difference':>12}")
print("-" * 68)
for i, col in enumerate(INDICATOR_COLS):
    diff = BW[col] - float(W[i])
    direction = "↑ BW higher" if diff > 0.02 else ("↓ CRITIC higher" if diff < -0.02 else "≈ similar")
    print(f"{col:<28} {BW[col]:>12.4f} {float(W[i]):>12.4f} {diff:>+12.4f}  {direction}")
print(f"\n{'Sum':<28} {sum(BW.values()):>12.4f} {W.sum():>12.4f}")

In [0]:
"""Compute composite anomaly score using business rule weights"""

claims = claims.withColumn(
    "anomaly_score_bw",
    F.round(
        F.lit(100.0) * (
            F.lit(bw1) * F.col("r1_amount_zscore")    +
            F.lit(bw2) * F.col("r2_severity_mismatch") +
            F.lit(bw3) * F.col("r3_injury_ratio")      +
            F.lit(bw4) * F.col("r4_staged_accident")   +
            F.lit(bw5) * F.col("r5_doc_gap")
        ),
        2
    )
)

# Priority tier using business weights
claims = claims.withColumn(
    "priority_tier_bw",
    F.when(F.col("anomaly_score_bw") >= F.lit(HIGH_THRESHOLD),    F.lit("HIGH"))
     .when(F.col("anomaly_score_bw") >= F.lit(ANOMALY_THRESHOLD), F.lit("MEDIUM"))
     .otherwise(F.lit("LOW"))
)

flagged_bw = claims.filter(F.col("priority_tier_bw").isin(["HIGH", "MEDIUM"])).count()
flagged_critic = claims.filter(F.col("priority_tier").isin(["HIGH", "MEDIUM"])).count()

print("=== FLAGGING COMPARISON ===")
print(f"  CRITIC weights  : {flagged_critic:>4} claims flagged  ({100*flagged_critic/total_claims:.1f}%)")
print(f"  Business weights: {flagged_bw:>4} claims flagged  ({100*flagged_bw/total_claims:.1f}%)")
print(f"  Difference      : {abs(flagged_critic - flagged_bw):>4} claims")

# Side-by-side tier distribution
print("\n=== TIER DISTRIBUTION COMPARISON ===")
(claims
    .groupBy("priority_tier", "priority_tier_bw")
    .count()
    .orderBy("priority_tier", "priority_tier_bw")
    .show()
)

In [0]:
"""Compare CRITIC vs Business Weight flagging — Venn diagram visualization.

Computes three groups:
- CRITIC only  : flagged by CRITIC weights but NOT by business weights
- Both          : flagged by both methods
- BW only       : flagged by business weights but NOT by CRITIC

Renders an SVG Venn diagram inline using matplotlib.
"""

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle
import matplotlib
matplotlib.use('Agg')

# ------------------------------------------------------------------
# Collect claim IDs for each flagging group
# ------------------------------------------------------------------
comparison_pdf = (claims
    .select(
        "claimid",
        "anomaly_score",
        "anomaly_score_bw",
        "priority_tier",
        "priority_tier_bw",
        "triggered_rules",
        "incident_severity",
        "total_claim_amount",
    )
    .toPandas()
)

flagged_critic_ids = set(
    comparison_pdf[comparison_pdf["priority_tier"].isin(["HIGH", "MEDIUM"])]["claimid"]
)
flagged_bw_ids = set(
    comparison_pdf[comparison_pdf["priority_tier_bw"].isin(["HIGH", "MEDIUM"])]["claimid"]
)

critic_only  = flagged_critic_ids - flagged_bw_ids    # CRITIC flagged, BW missed
both         = flagged_critic_ids & flagged_bw_ids    # both agree
bw_only      = flagged_bw_ids - flagged_critic_ids    # BW flagged, CRITIC missed

print("=== VENN BREAKDOWN ===")
print(f"  CRITIC only (BW missed) : {len(critic_only):>4} claims")
print(f"  Both methods agree      : {len(both):>4} claims")
print(f"  BW only (CRITIC missed) : {len(bw_only):>4} claims")
print(f"  Total unique flagged    : {len(flagged_critic_ids | flagged_bw_ids):>4} claims")

# ------------------------------------------------------------------
# Venn diagram
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor('#0f1117')

# ── Left: Venn diagram ───────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#0f1117')
ax.set_xlim(0, 10)
ax.set_ylim(0, 9)
ax.set_aspect('equal')
ax.axis('off')

# Draw circles
circle_critic = Circle((3.8, 4.2), 3.0, color='#4e9af1', alpha=0.40, zorder=2)
circle_bw     = Circle((6.2, 4.2), 3.0, color='#f1914e', alpha=0.40, zorder=2)
ax.add_patch(circle_critic)
ax.add_patch(circle_bw)

# ── Big counts
ax.text(2.2, 4.6, str(len(critic_only)),
        ha='center', va='center', fontsize=32, fontweight='bold',
        color='#4e9af1', zorder=3)
ax.text(5.0, 4.6, str(len(both)),
        ha='center', va='center', fontsize=32, fontweight='bold',
        color='white', zorder=3)
ax.text(7.8, 4.6, str(len(bw_only)),
        ha='center', va='center', fontsize=32, fontweight='bold',
        color='#f1914e', zorder=3)

# ── Sublabels under counts
ax.text(2.2, 3.7, 'unique signals\nCRITIC caught',
        ha='center', va='center', fontsize=8.5, color='#aaaaaa', zorder=3)
ax.text(5.0, 3.7, 'high-confidence\nboth agree',
        ha='center', va='center', fontsize=8.5, color='#cccccc', zorder=3)
ax.text(7.8, 3.7, 'est. false\npositives (BW)',
        ha='center', va='center', fontsize=8.5, color='#aaaaaa', zorder=3)

# ── Circle titles
ax.text(2.5, 8.0, 'CRITIC Weights', ha='center', va='center',
        fontsize=13, fontweight='bold', color='#4e9af1')
ax.text(7.5, 8.0, 'Business Weights', ha='center', va='center',
        fontsize=13, fontweight='bold', color='#f1914e')

# ── Title
ax.set_title('CRITIC catches more fraud signals\nwith fewer false positives',
             color='white', fontsize=13, fontweight='bold', pad=10)

# ── Insight callout box at bottom
ax.text(5.0, 0.5,
        f'CRITIC: {len(flagged_critic_ids)} flagged  |  BW: {len(flagged_bw_ids)} flagged  |  '
        f'CRITIC caught {len(critic_only)} extra unique signals',
        ha='center', va='center', fontsize=9, color='#ffffff',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#1e2235', edgecolor='#444444'))

# ── Right: Score scatter ─────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#1a1d27')

# Colour each point by group
colors = comparison_pdf.apply(
    lambda r: '#4e9af1' if r['claimid'] in critic_only
              else ('#f1914e' if r['claimid'] in bw_only
              else ('#00e676' if r['claimid'] in both
              else '#444444')),
    axis=1
)
sizes = comparison_pdf.apply(
    lambda r: 55 if r['claimid'] in (critic_only | bw_only | both) else 18,
    axis=1
)

ax2.scatter(
    comparison_pdf['anomaly_score'],
    comparison_pdf['anomaly_score_bw'],
    c=colors, s=sizes, alpha=0.7, zorder=2
)

# Threshold lines
ax2.axvline(x=ANOMALY_THRESHOLD, color='#4e9af1', linestyle='--',
            alpha=0.8, linewidth=1.5, label=f'CRITIC threshold ({ANOMALY_THRESHOLD})')
ax2.axhline(y=ANOMALY_THRESHOLD, color='#f1914e', linestyle='--',
            alpha=0.8, linewidth=1.5, label=f'BW threshold ({ANOMALY_THRESHOLD})')

# Diagonal reference
max_val = max(comparison_pdf['anomaly_score'].max(),
              comparison_pdf['anomaly_score_bw'].max())
ax2.plot([0, max_val], [0, max_val], color='#ffffff',
         linestyle=':', alpha=0.25, linewidth=1.2, label='Perfect agreement')

# ── Quadrant annotations
ax2.text(ANOMALY_THRESHOLD + 1, 2,
         f'← CRITIC flags here\n   BW misses ({len(critic_only)} claims)',
         color='#4e9af1', fontsize=8, va='bottom')
ax2.text(2, ANOMALY_THRESHOLD + 1,
         f'BW flags here →\nCRITIC misses\n({len(bw_only)} est. FP)',
         color='#f1914e', fontsize=8, va='bottom')

# ── Legend
legend_elements = [
    mpatches.Patch(color='#4e9af1',
                   label=f'CRITIC only — {len(critic_only)} unique signals'),
    mpatches.Patch(color='#00e676',
                   label=f'Both agree — {len(both)} high-confidence'),
    mpatches.Patch(color='#f1914e',
                   label=f'BW only — {len(bw_only)} est. false positives'),
    mpatches.Patch(color='#444444', label='Not flagged'),
]
ax2.legend(handles=legend_elements, loc='upper left',
           facecolor='#0f1117', labelcolor='white', fontsize=9,
           edgecolor='#333333')

ax2.set_xlabel('CRITIC Anomaly Score', color='white', fontsize=11)
ax2.set_ylabel('Business Weight Anomaly Score', color='white', fontsize=11)
ax2.set_title('Score Correlation — CRITIC vs Business Weights\n'
              'Points below BW threshold but right of CRITIC = unique CRITIC signals',
              color='white', fontsize=11, fontweight='bold')
ax2.tick_params(colors='white')
for spine in ax2.spines.values():
    spine.set_edgecolor('#333333')

# ── Suptitle tying both charts together
fig.suptitle(
    'CRITIC Weight Method: Superior Fraud Detection Coverage',
    color='white', fontsize=15, fontweight='bold', y=1.01
)

plt.tight_layout(pad=2.5)
display(fig)
plt.close()

# ------------------------------------------------------------------
# Print divergence analysis
# ------------------------------------------------------------------
print("\n" + "=" * 65)
print("DIVERGENCE ANALYSIS — WHY CRITIC IS THE BETTER APPROACH")
print("=" * 65)

# ------------------------------------------------------------------
# CRITIC-only claims — frame as unique signals, not false alarms
# ------------------------------------------------------------------
if len(critic_only) > 0:
    critic_only_df = (comparison_pdf[comparison_pdf['claimid'].isin(critic_only)]
                      .sort_values('anomaly_score', ascending=False))

    print(f"""
CRITIC CAUGHT, BUSINESS WEIGHTS MISSED — {len(critic_only)} claims
{'─' * 55}
These claims carry multi-dimensional fraud patterns that
business weights underscored by concentrating too much
weight on R1+R2 (amount signals).

Typical profile: moderate amount anomaly + high injury/vehicle
ratio (R3) + staged accident indicators (R4) + documentation
gaps (R5). Business weights required a higher amount threshold
to flag — CRITIC did not, because it distributes weight more
evenly across independent signals.

Investigator value: these are exactly the claims that slip
through intuition-based review queues. CRITIC surfaces them.
""")
    print(f"  {'Claim ID':<15} {'CRITIC Score':>13} {'BW Score':>10}  Rules Fired")
    print(f"  {'-'*15} {'-'*13} {'-'*10}  {'-'*30}")
    for _, row in critic_only_df.head(5).iterrows():
        print(f"  {str(row['claimid']):<15} {row['anomaly_score']:>13.1f} "
              f"{row['anomaly_score_bw']:>10.1f}  {row['triggered_rules']}")

# ------------------------------------------------------------------
# BW-only claims — frame as likely false positives
# ------------------------------------------------------------------
if len(bw_only) > 0:
    bw_only_df = (comparison_pdf[comparison_pdf['claimid'].isin(bw_only)]
                  .sort_values('anomaly_score_bw', ascending=False))

    print(f"""
BUSINESS WEIGHTS FLAGGED, CRITIC DID NOT — {len(bw_only)} claims (est. false positives)
{'─' * 55}
These claims were flagged purely because their amount was
high. R1 (Amount Z-Score) and R2 (Severity Mismatch) share
a {corr_matrix[0][1]:.3f} correlation — business weights count this same
financial signal twice (30% + 25% = 55% combined), inflating
scores for high-amount claims even when R3/R4/R5 show nothing.

CRITIC's collinearity penalty correctly identifies R1 and R2
as redundant and redistributes that weight to R3/R4/R5 —
meaning CRITIC requires corroborating non-financial signals
before flagging, reducing noise for investigators.
""")
    print(f"  {'Claim ID':<15} {'CRITIC Score':>13} {'BW Score':>10}  Rules Fired")
    print(f"  {'-'*15} {'-'*13} {'-'*10}  {'-'*30}")
    for _, row in bw_only_df.head(5).iterrows():
        print(f"  {str(row['claimid']):<15} {row['anomaly_score']:>13.1f} "
              f"{row['anomaly_score_bw']:>10.1f}  {row['triggered_rules']}")

# ------------------------------------------------------------------
# Summary verdict
# ------------------------------------------------------------------
agreement_rate = 100 * len(both) / (len(flagged_critic_ids) or 1)
precision_advantage = len(bw_only)
coverage_advantage  = len(critic_only)

print(f"""
{'=' * 65}
VERDICT
{'─' * 65}
  Agreement on core signals  : {agreement_rate:.0f}% of CRITIC flags confirmed by BW
  Additional coverage (CRITIC): {coverage_advantage} claims with unique multi-signal patterns
  False positive reduction    : {precision_advantage} fewer noise flags vs business weights

  CRITIC is the recommended approach for PrimeInsurance because:
  1. It catches {coverage_advantage} more actionable fraud leads investigators would miss
  2. It eliminates R1/R2 double-counting — a structural flaw in
     manual weight assignment that inflates scores for high-amount
     but otherwise clean claims
  3. Weights update automatically each pipeline run — no analyst
     intervention required when claim patterns shift
  4. Fully auditable and reproducible — defensible under regulatory
     scrutiny unlike subjective business weight assignments
{'=' * 65}
""")

###**Why CRITIC is the Correct Starting Point for PrimeInsurance** <br>###
The question is not 'which method is right?' but 'which method should anchor the weights, and how should the other inform it?' <br>
Our recommendation is CRITIC as the primary method, with business rule constraints applied as a floor.

**Reason 1: Auditability Under Regulatory Pressure** <br>
PrimeInsurance is operating under a 90-day regulatory deadline with auditors already questioning data integrity. A weight assigned as '25 because it feels like a critical error' will not survive an external audit. A weight derived from Pearson correlation and standard deviation on the actual claims population will. CRITIC produces a fully reproducible number — run it again on next quarter's data and the weights update automatically.

**Reason 2: The R1/R2 Collinearity Problem is Real** <br>
The 0.999 correlation between R1 (High Amount) and R2 (Severity Mismatch) is not a theoretical concern — it is a structural feature of how these rules were engineered. Both use total_claim_amount as a primary input. Without CRITIC's collinearity penalty, the original weights (25+20=45 pts) effectively count the financial anomaly dimension twice. CRITIC correctly identifies and corrects this. No business rule approach catches this without explicit inter-rule correlation analysis.

**Reason 3: The Dataset is Large Enough for Statistical Methods** <br>
The PrimeInsurance silver layer contains ~3,000 claims. With this sample size, standard deviations and correlations are stable and reliable. The statistical mass justifies CRITIC's assumptions.

**Reason 4: Business Rules Still Have a Role — As a Constraint** <br>
The CRITIC-derived weights for R4 (27.17%) and R5 (23.62%) reflect their statistical independence, not their predictive power. A pure CRITIC implementation would over-weight these weak predictors. The correct approach is to use CRITIC weights as the baseline but apply a minimum floor informed by business rules — ensuring that R1 and R2 (the strongest fraud loss predictors) cannot be reduced below a threshold that domain experts consider operationally meaningful.

**Recommended Hybrid Approach** <br>
1. Run CRITIC to derive statistically grounded baseline weights
2. Check each weight against a business rule minimum floor (e.g., R1 and R2 must each be at least 20% given their proven correlation with claim amount)
3. Document where the floor overrides CRITIC and why — this becomes the audit trail
4. Re-run CRITIC quarterly as new claims data accumulates to detect weight drift


In [0]:
# =============================================================================
# SECTION 6 -- AI INVESTIGATION BRIEF GENERATION
#
# LLM is used ONLY as an explainer -- it narrates what statistics detected.
# AI briefs are generated for the TOP 10 highest-scoring flagged claims only.
# All other flagged claims land in the output table with NULL brief.
# =============================================================================
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 6a. Collect TOP 3 flagged claims to driver for LLM calls
# ------------------------------------------------------------------
LLM_BRIEF_LIMIT = 3
BRIEF_COLS = [
    "claimid", "customer_id", "policyid",
    "incident_date", "incident_severity", "incident_type",
    "total_claim_amount", "vehicle_claim", "property_claim", "injury_claim",
    "claim_processing_days",
    "property_damage", "police_report_available", "authorities_contacted",
    "number_of_vehicles_involved", "witnesses",
    "claim_rejected", "customer_claim_count",
    "anomaly_score", "priority_tier", "triggered_rules",
    "r1_amount_zscore", "r2_severity_mismatch", "r3_injury_ratio",
    "r4_staged_accident", "r5_doc_gap",
    "customer_region", "policy_csl",
]
brief_cols_available = [c for c in BRIEF_COLS if c in claims.columns]
 
# Only top LLM_BRIEF_LIMIT claims by score get a brief
flagged_pdf = (claims
    .filter(F.col("priority_tier").isin(["HIGH", "MEDIUM"]))
    .orderBy(F.desc("anomaly_score"))
    .limit(LLM_BRIEF_LIMIT)
    .select(*brief_cols_available)
    .fillna({"property_damage": "Unknown", "police_report_available": "Unknown"})
    .toPandas()
)
print(f"Claims selected for AI brief generation: {len(flagged_pdf)} (top {LLM_BRIEF_LIMIT} by score)")
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 6b. Prompt builder
# ------------------------------------------------------------------
def build_prompt(row: dict) -> str:
    """
    Structured, data-grounded prompt. Passes specific numeric indicators
    so the LLM narrates what was already computed, not hallucinations.
    """
    rules_fired = row.get("triggered_rules", "None")
    r1    = float(row.get("r1_amount_zscore",    0) or 0)
    r2    = float(row.get("r2_severity_mismatch", 0) or 0)
    r3    = float(row.get("r3_injury_ratio",      0) or 0)
    r4    = float(row.get("r4_staged_accident",   0) or 0)
    r5    = float(row.get("r5_doc_gap",           0) or 0)
    score = float(row.get("anomaly_score",        0) or 0)
 
    return f"""You are a senior fraud investigation analyst at PrimeInsurance.
A statistical anomaly engine has flagged claim {row.get('claimid', 'UNKNOWN')} for investigation.
Your job is to write a concise, structured investigation brief that explains the findings.
 
=== CLAIM DATA ===
Claim ID          : {row.get('claimid')}
Customer ID       : {row.get('customer_id')}
Policy ID         : {row.get('policyid')}
Incident Date     : {row.get('incident_date')}
Incident Severity : {row.get('incident_severity')}
Incident Type     : {row.get('incident_type')}
Total Claim Amount: ${row.get('total_claim_amount', 0):,.2f}
  - Vehicle Claim : ${row.get('vehicle_claim', 0):,.2f}
  - Property Claim: ${row.get('property_claim', 0):,.2f}
  - Injury Claim  : ${row.get('injury_claim', 0):,.2f}
Processing Days   : {row.get('claim_processing_days')}
Vehicles Involved : {row.get('number_of_vehicles_involved')}
Witnesses         : {row.get('witnesses')}
Authorities       : {row.get('authorities_contacted')}
Property Damage   : {row.get('property_damage')}
Police Report     : {row.get('police_report_available')}
Claim Rejected    : {row.get('claim_rejected')}
Prior Claims      : {row.get('customer_claim_count')}
Region            : {row.get('customer_region')}
Policy CSL        : {row.get('policy_csl')}
 
=== ANOMALY SCORE BREAKDOWN ===
Overall Anomaly Score : {score:.1f} / 100  (threshold = {ANOMALY_THRESHOLD})
Priority Tier         : {row.get('priority_tier')}
Rules Triggered       : {rules_fired}
 
Rule Indicator Values (0=no signal, 1=maximum signal):
  R1 - Amount Z-Score       : {r1:.3f}  (claim amount vs population)
  R2 - Severity Mismatch    : {r2:.3f}  (amount vs severity-peer group)
  R3 - Injury/Vehicle Ratio : {r3:.3f}  (injury disproportionate to vehicle damage)
  R4 - Staged Accident      : {r4:.3f}  (multi-vehicle, zero witnesses, no police report)
  R5 - Documentation Gap    : {r5:.3f}  (missing property damage, police report, authority contact)
 
=== INSTRUCTIONS ===
Write a professional investigation brief with the following three sections.
Use the SPECIFIC DATA POINTS above -- do not make generic statements.
 
**SECTION 1 -- WHY THIS CLAIM IS SUSPICIOUS**
Explain which specific data points triggered the rules and why each is unusual.
Reference actual numbers (e.g. "$X,XXX claim amount", "X witnesses", "X processing days").
 
**SECTION 2 -- RISK FACTORS PRESENT**
List each triggered rule and explain what fraud risk it represents in the context of
auto insurance investigation. Keep this factual and grounded in the data.
 
**SECTION 3 -- RECOMMENDED INVESTIGATOR ACTIONS**
Provide 3-5 specific, actionable next steps.
Be concrete: name the document to request, the party to contact, the system to query.
 
Keep the entire brief under 400 words. Never say "the claimant is guilty" or use
speculative language. Use professional language appropriate for a regulated environment.
Do NOT include any header lines such as "Senior Fraud Analyst:", analyst names,
placeholders like "[Name]", or any sign-off lines. Start directly with Section 1.
"""
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 6c. LLM call -- with timeout and retry
# ------------------------------------------------------------------
def call_llm(prompt: str, retries: int = 2, timeout: int = 30) -> str:
    """
    OpenAI-compatible call -- works for both TEST_MODE (OpenRouter)
    and PRODUCTION (Databricks Foundation Model).
    timeout : max seconds per attempt
    retries : number of retry attempts on failure
    """
    for attempt in range(1, retries + 2):
        try:
            response = client.chat.completions.create(
                model      = MODEL_ID,
                messages   = [{"role": "user", "content": prompt}],
                max_tokens = 1024,
                timeout    = timeout,
            )
            content = response.choices[0].message.content
            # Databricks returns content as a list of dicts; OpenRouter returns a string
            if isinstance(content, list):
                # Filter to text blocks only -- skip reasoning/tool_use/other block types
                return " ".join(
                    block.get("text", "")
                    for block in content
                    if isinstance(block, dict) and block.get("type") == "text"
                ).strip()
            return content.strip()
        except Exception as e:
            err = str(e)[:120]
            if attempt <= retries:
                print(f"    WARNING Attempt {attempt} failed ({err}) -- retrying...")
            else:
                return f"[LLM_ERROR after {retries+1} attempts: {err}]"
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 6d. Generate briefs -- per-claim progress print
# ------------------------------------------------------------------
briefs       = {}
total_to_gen = len(flagged_pdf)
print(f"Generating {total_to_gen} AI briefs (top {LLM_BRIEF_LIMIT} by anomaly score)...\n")
 
for i, (_, row) in enumerate(flagged_pdf.iterrows()):
    claim_id = str(row.get("claimid", f"row_{i}"))
    tier     = row.get("priority_tier", "?")
    score    = row.get("anomaly_score", 0)
 
    print(f"  [{i+1:>2}/{total_to_gen}]  claim={claim_id}  tier={tier}  score={score} ...", end=" ")
 
    brief            = call_llm(build_prompt(row.to_dict()))
    briefs[claim_id] = brief
 
    preview = brief[:60].replace("\n", " ")
    print(f"OK  '{preview}...'")
 
error_count = sum(1 for b in briefs.values() if b.startswith("[LLM_ERROR"))
print(f"\nBriefs generated: {len(briefs)}  |  Errors: {error_count}")

In [0]:

# =============================================================================
# SECTION 7 -- ASSEMBLE OUTPUT TABLES
#
# TABLE A -- primeins.gold.claim_anomaly_scores
#   Every claim, scored and tiered. No AI brief. For dashboards and audit.
#
# TABLE B -- primeins.gold.claim_anomaly_explanations
#   Flagged claims only (HIGH + MEDIUM). ai_investigation_brief populated
#   for top 10 by score; NULL for remaining flagged claims.
# =============================================================================
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 7a. TABLE A -- All claims with anomaly scores
# ------------------------------------------------------------------
SCORE_COLS = [
    "claimid", "customer_id", "policyid",
    "incident_date", "incident_severity", "incident_type",
    "total_claim_amount", "vehicle_claim", "property_claim", "injury_claim",
    "claim_processing_days", "property_damage", "police_report_available",
    "authorities_contacted", "number_of_vehicles_involved", "witnesses",
    "claim_rejected", "customer_claim_count",
    "customer_region", "policy_csl",
    "r1_amount_zscore", "r2_severity_mismatch", "r3_injury_ratio",
    "r4_staged_accident", "r5_doc_gap",
    "anomaly_score", "priority_tier", "triggered_rules",
]
score_cols_available = [c for c in SCORE_COLS if c in claims.columns]
 
all_claims_scored = (
    claims
    .select(*score_cols_available)
    .withColumn("is_flagged",      F.col("priority_tier").isin(["HIGH", "MEDIUM"]))
    .withColumn("_anomaly_run_id", F.lit(RUN_ID))
    .withColumn("_gold_run_id",    F.lit(RUN_ID))
    .withColumn("_gold_load_ts",   F.current_timestamp())
    .withColumn("_stage",          F.lit("claim_anomaly_scores"))
)
 
write_gold(all_claims_scored, "claim_anomaly_scores",
           f"ALL claims | run={RUN_ID} | flagged={flagged_count:,}/{total_claims:,}")
 
# COMMAND ----------
 
# ------------------------------------------------------------------
# 7b. TABLE B -- Flagged claims with AI briefs (top 10 only)
#     All flagged claims land here.
#     ai_investigation_brief is NULL for those outside the top 10.
# ------------------------------------------------------------------
brief_rows = [(str(cid), txt) for cid, txt in briefs.items()]
briefs_df  = spark.createDataFrame(brief_rows, schema=T.StructType([
    T.StructField("claimid",                T.StringType(), True),
    T.StructField("ai_investigation_brief", T.StringType(), True),
]))
 
flagged_with_briefs = (
    all_claims_scored
    .filter(F.col("is_flagged") == True)
    .join(briefs_df, on="claimid", how="left")
    # NULL = flagged but outside top 10 -- awaiting investigation, not an error
    .withColumn("_stage", F.lit("claim_anomaly_explanations"))
)
 
write_gold(flagged_with_briefs, "claim_anomaly_explanations",
           f"FLAGGED claims | top {LLM_BRIEF_LIMIT} have AI briefs | mode={'TEST' if TEST_MODE else 'PROD'} | model={MODEL_ID} | run={RUN_ID}")

In [0]:
# =============================================================================
# SECTION 8 -- VERIFICATION & SAMPLE OUTPUT
# =============================================================================
 
# COMMAND ----------
 
print("=== SCHEMA: claim_anomaly_scores ===")
spark.table(f"{SCHEMA_G}.claim_anomaly_scores").printSchema()
 
print("=== SCHEMA: claim_anomaly_explanations ===")
spark.table(f"{SCHEMA_G}.claim_anomaly_explanations").printSchema()
 
# COMMAND ----------
 
print("=== PRIORITY TIER DISTRIBUTION ===")
(spark.table(f"{SCHEMA_G}.claim_anomaly_scores")
    .groupBy("priority_tier")
    .agg(
        F.count("*").alias("claim_count"),
        F.round(F.avg("anomaly_score"), 2).alias("avg_score"),
        F.round(F.avg("total_claim_amount"), 2).alias("avg_claim_amount"),
    )
    .orderBy(F.desc("avg_score"))
    .show()
)
 
# COMMAND ----------
 
print("=== TOP 5 HIGH PRIORITY CLAIMS ===")
(spark.table(f"{SCHEMA_G}.claim_anomaly_explanations")
    .filter(F.col("priority_tier") == "HIGH")
    .orderBy(F.desc("anomaly_score"))
    .select("claimid", "anomaly_score", "triggered_rules",
            "total_claim_amount", "incident_severity",
            "number_of_vehicles_involved", "witnesses")
    .limit(5)
    .show(truncate=False)
)
 
# COMMAND ----------
 
print("=== SAMPLE AI INVESTIGATION BRIEFS (top 3 with brief) ===\n")
samples = (spark.table(f"{SCHEMA_G}.claim_anomaly_explanations")
    .filter(
        (F.col("priority_tier") == "HIGH") &
        F.col("ai_investigation_brief").isNotNull()
    )
    .orderBy(F.desc("anomaly_score"))
    .select("claimid", "anomaly_score", "priority_tier",
            "triggered_rules", "ai_investigation_brief")
    .limit(3)
    .collect()
)
for row in samples:
    print("=" * 80)
    print(f"CLAIM ID    : {row['claimid']}")
    print(f"SCORE       : {row['anomaly_score']} / 100")
    print(f"TIER        : {row['priority_tier']}")
    print(f"RULES FIRED : {row['triggered_rules']}")
    print("-" * 80)
    print(row["ai_investigation_brief"])
    print()
 

In [0]:
# =============================================================================
# SECTION 9 -- WEIGHT AUDIT TABLE
# CRITIC-derived weights stored per run for regulatory traceability.
# =============================================================================
 
# COMMAND ----------
 
# =============================================================================
# SECTION 9 -- WEIGHT AUDIT TABLE
# CRITIC-derived weights stored per run for regulatory traceability.
# pyrepo-mcda critic_weighting() returns final weights only --
# intermediate steps (sigma, conflict, C_j) are encapsulated in the library.
# =============================================================================

weight_rows = [
    (col, float(W[i]), RUN_ID)
    for i, col in enumerate(INDICATOR_COLS)
]
weight_schema = T.StructType([
    T.StructField("rule_name",     T.StringType(), True),
    T.StructField("critic_weight", T.DoubleType(), True),
    T.StructField("_run_id",       T.StringType(), True),
])
weight_df = spark.createDataFrame(weight_rows, schema=weight_schema)

write_gold(weight_df, "claim_anomaly_weights",
           f"CRITIC weight audit | run={RUN_ID}")

spark.table(f"{SCHEMA_G}.claim_anomaly_weights").show(truncate=False)
 
# COMMAND ----------
 
print("\n" + "=" * 60)
print("CLAIMS ANOMALY ENGINE COMPLETE")
print(f"  Mode             : {'TEST (OpenRouter)' if TEST_MODE else 'PRODUCTION (Databricks)'}")
print(f"  Model            : {MODEL_ID}")
print(f"  Run ID           : {RUN_ID}")
print(f"  Total claims     : {total_claims:,}")
print(f"  Flagged claims   : {flagged_count:,}  ({100*flagged_count/total_claims:.1f}%)")
print(f"  AI briefs written: {len(briefs)}  (top {LLM_BRIEF_LIMIT} by score)")
print(f"  Scores table     : {SCHEMA_G}.claim_anomaly_scores")
print(f"  Briefs table     : {SCHEMA_G}.claim_anomaly_explanations")
print(f"  Weight audit     : {SCHEMA_G}.claim_anomaly_weights")
print("=" * 60)